# Demo 02 — Industrial Shortcut Trap: When The Classifier Learns The Marker, Not The Defect

**Demo promise:** We will show a model that performs well when a coloured side-band marker agrees with the label, collapses when that marker is swapped or removed, exposes the shortcut through exact counterfactuals and ablations, and improves under marker-randomised mitigation without turning the audit into a solved problem.

In real inspection pipelines, labels can accidentally correlate with camera marks, fixture positions, stamps, borders, overlays, or preprocessing artefacts. This notebook uses **real NEU shortcut data only** and a controlled side-band marker nuisance to show how a learned classifier can take that shortcut.

## Learning goals
- show the real industrial defect task before introducing the nuisance artefact;
- train a credible learned baseline on real NEU images with frozen pretrained features;
- compare aligned validation with aligned, swapped-marker, and no-marker audits;
- use exact marker counterfactuals, region ablations, occlusion, and exemplar retrieval to inspect the shortcut;
- train a marker-randomised mitigation and re-test the same failure case.

## Why this demo matters
The lesson is not that industrial shortcuts are rare. The lesson is that they are easy to create accidentally, easy to trust when metrics look fine, and only safe when they are audited explicitly. A coloured acquisition strip can become as dangerous as a natural background shortcut if the pipeline lets it correlate with the label.


### Section 0 — Runtime and data checks
The next cell discovers the repository root, configures a notebook-safe plotting backend, resolves the required NEU shortcut manifest, locks the seed, and prints the first visible run-status lines. There is no fallback dataset path in this notebook.


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import tempfile
import textwrap
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Sequence

# Keep notebook caches writable even when the kernel starts from a nested folder.
_cache_root = Path(tempfile.gettempdir()) / 'xai_notebook_cache'
_cache_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('XDG_CACHE_HOME', str(_cache_root))
os.environ.setdefault('MPLCONFIGDIR', str(_cache_root / 'matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import matplotlib
from IPython import get_ipython

_ipython = get_ipython()
if _ipython is None or 'IPKernelApp' not in getattr(_ipython, 'config', {}):
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFilter, ImageOps
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms.functional import pil_to_tensor

warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive, and thus cannot be shown')

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

NOTEBOOK_RELATIVE_PATH = Path('notebooks/shortcut_lab/02_industrial_shortcut_trap.ipynb')
MANIFEST_RELATIVE_PATH = Path('data/processed/neu_cls/shortcut_binary/manifest.jsonl')
DATA_MODE = 'real_neu_controlled_shortcut'
ANALYSIS_SIZE = (224, 224)
LABEL_NAMES = ['linear_defect', 'area_defect']
LABEL_TO_INT = {label: index for index, label in enumerate(LABEL_NAMES)}
INT_TO_LABEL = {value: key for key, value in LABEL_TO_INT.items()}
STAMP_COLOUR = {
    'blue': (56, 116, 214),
    'red': (216, 70, 64),
    'none': (128, 128, 128),
}
STAMP_DISPLAY_NAME = {'blue': 'Blue side-band marker', 'red': 'Red side-band marker', 'none': 'No marker'}
COMMON_COLOUR = '#4c78a8'
CROSSED_COLOUR = '#e45756'
CORRECT_COLOUR = '#2b8a3e'
WRONG_COLOUR = '#c92a2a'
STAMP_COLOUR_HEX = {'blue': '#3874d6', 'red': '#d84640'}
FORBIDDEN_TOY_PHRASES = [
    'render_' + 'panel',
    'make_' + 'samples',
    'stamp_' + 'shortcut_score',
    'shape_' + 'score',
    'Part' + 'Sample',
    'synthetic ' + 'part',
]
REPRESENTATIVE_BASE_IDS = {
    'linear_defect': 'sc_000',
    'area_defect': 'in_000',
}

plt.rcParams.update({
    'figure.dpi': 140,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'font.size': 11,
})


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        manifest = candidate / MANIFEST_RELATIVE_PATH
        if manifest.exists():
            return candidate
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find the repository root or the NEU shortcut manifest. '
        'Run this notebook from the repository, or prepare the NEU shortcut data first.'
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_PATH = PROJECT_ROOT / NOTEBOOK_RELATIVE_PATH
MANIFEST_PATH = PROJECT_ROOT / MANIFEST_RELATIVE_PATH
MANIFEST_EXISTS = MANIFEST_PATH.exists()

if not MANIFEST_EXISTS:
    raise FileNotFoundError(
        'Real NEU shortcut data is required for this notebook. '
        'Prepare data/processed/neu_cls/shortcut_binary/manifest.jsonl and rerun.'
    )

print(f'DATA_MODE: {DATA_MODE}')
print(f'manifest_exists: {MANIFEST_EXISTS}')
print(f'manifest_path: {MANIFEST_PATH}')
print(f'project_root: {PROJECT_ROOT}')
print(f'seed: {SEED}')


## Dataset and task definition
### Section 1 — Parse the real NEU shortcut manifest
We normalise the manifest into typed records, resolve image paths relative to the repository root, validate that the required real files exist, and make the stamp metadata explicit so later probes can use the same region consistently.


In [ ]:
@dataclass(frozen=True)
class BoundingBox:
    x: int
    y: int
    width: int
    height: int


@dataclass(frozen=True)
class NEUShortcutRecord:
    sample_id: str
    base_id: str
    image_path: str
    split: str
    label: str
    label_int: int
    variant: str
    stamp: str
    is_aligned: bool
    stamp_region: BoundingBox
    object_region: BoundingBox


manifest_rows = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding='utf-8').splitlines()]
records: list[NEUShortcutRecord] = []
missing_paths: list[str] = []
for raw in manifest_rows:
    raw_path = Path(raw['image_path'])
    image_path = raw_path if raw_path.is_absolute() else PROJECT_ROOT / raw_path
    sample_id = raw.get('sample_id') or raw_path.with_suffix('').as_posix()
    stem = raw_path.stem
    label = raw['label']
    variant = raw['variant']
    variant_suffix = f'_{variant}'
    base_id = stem[:-len(variant_suffix)] if stem.endswith(variant_suffix) else stem.rsplit('_', 1)[0]
    stamp = raw['stamp']
    assert label in LABEL_TO_INT
    assert variant in {'correlated', 'clean', 'swapped_stamp', 'no_stamp'}
    assert stamp in {'blue', 'red', 'none'}
    if not image_path.exists():
        missing_paths.append(str(image_path))
    records.append(
        NEUShortcutRecord(
            sample_id=sample_id,
            base_id=base_id,
            image_path=str(image_path.resolve()),
            split=raw['split'],
            label=label,
            label_int=LABEL_TO_INT[label],
            variant=variant,
            stamp=stamp,
            is_aligned=bool(raw['is_aligned']),
            stamp_region=BoundingBox(**raw['stamp_region']),
            object_region=BoundingBox(**raw['object_region']),
        )
    )

assert not missing_paths, f'Manifest entries point to missing files: {missing_paths[:5]}'
neu_shortcut = pd.DataFrame(asdict(record) for record in records)
neu_shortcut['stamp_region'] = neu_shortcut['stamp_region'].map(lambda raw: raw if isinstance(raw, BoundingBox) else BoundingBox(**raw))
neu_shortcut['object_region'] = neu_shortcut['object_region'].map(lambda raw: raw if isinstance(raw, BoundingBox) else BoundingBox(**raw))
assert neu_shortcut['sample_id'].is_unique
label_names = [label for label in LABEL_NAMES if label in set(neu_shortcut['label'].unique())]
assert set(label_names) == set(LABEL_NAMES)

run_summary = pd.DataFrame([
    {'Field': 'Mode', 'Value': DATA_MODE},
    {'Field': 'Manifest exists', 'Value': str(MANIFEST_EXISTS)},
    {'Field': 'Manifest path', 'Value': MANIFEST_PATH.as_posix()},
    {'Field': 'Project root', 'Value': PROJECT_ROOT.as_posix()},
    {'Field': 'Train records', 'Value': str(int((neu_shortcut['split'] == 'train').sum()))},
    {'Field': 'Test records', 'Value': str(int((neu_shortcut['split'] == 'test').sum()))},
    {'Field': 'Labels', 'Value': ', '.join(label_names)},
    {'Field': 'Missing files', 'Value': str(len(missing_paths))},
])
display(run_summary)

manifest_count_table = (
    neu_shortcut.groupby(['split', 'label', 'variant'])
    .size()
    .rename('count')
    .reset_index()
    .pivot(index=['split', 'label'], columns='variant', values='count')
    .fillna(0)
    .astype(int)
)
display(manifest_count_table)

train_correlated = neu_shortcut.loc[
    neu_shortcut['split'].eq('train') & neu_shortcut['variant'].eq('correlated')
].copy()
label_to_stamp = {
    label: train_correlated.loc[train_correlated['label'].eq(label), 'stamp'].mode().iloc[0]
    for label in LABEL_NAMES
}
stamp_to_opposite = {'blue': 'red', 'red': 'blue'}


### Section 2 — What is the real inspection task?
The first figure shows the real defect classification task before we lean on the acquisition artefact. Here we use the stamp-free `no_stamp` test images so the reader sees the defect content without the nuisance.

Waterbirds used natural context. This demo uses a controlled industrial nuisance. The label still comes from the defect pattern in the steel surface, not from the coloured side-band marker.


In [ ]:
DISPLAY_SIZE = ANALYSIS_SIZE


def load_rgb_image(path: str | Path) -> Image.Image:
    return Image.open(path).convert('RGB')


def make_model_canvas(image: Image.Image) -> Image.Image:
    return ImageOps.fit(image.convert('RGB'), ANALYSIS_SIZE, method=Image.Resampling.BICUBIC)


def prepare_display_image(path_or_image: str | Path | Image.Image, *, size: tuple[int, int] = DISPLAY_SIZE) -> Image.Image:
    image = load_rgb_image(path_or_image) if not isinstance(path_or_image, Image.Image) else path_or_image
    canvas = make_model_canvas(image)
    if size == ANALYSIS_SIZE:
        return canvas
    return canvas.resize(size, resample=Image.Resampling.BICUBIC)


def box_to_canvas(box: BoundingBox, *, source_size: tuple[int, int], target_size: tuple[int, int] = ANALYSIS_SIZE) -> BoundingBox:
    source_width, source_height = source_size
    target_width, target_height = target_size
    return BoundingBox(
        x=round(box.x * target_width / source_width),
        y=round(box.y * target_height / source_height),
        width=max(1, round(box.width * target_width / source_width)),
        height=max(1, round(box.height * target_height / source_height)),
    )


def crop_box(image: Image.Image, box: BoundingBox, *, pad: int = 14) -> Image.Image:
    left = max(0, box.x - pad)
    top = max(0, box.y - pad)
    right = min(image.size[0], box.x + box.width + pad)
    bottom = min(image.size[1], box.y + box.height + pad)
    return image.crop((left, top, right, bottom))


def representative_no_stamp_row(frame: pd.DataFrame, label: str) -> pd.Series:
    candidates = frame.loc[
        frame['split'].eq('test') & frame['variant'].eq('no_stamp') & frame['label'].eq(label)
    ].copy()
    base_id_override = REPRESENTATIVE_BASE_IDS.get(label)
    if base_id_override and base_id_override in set(candidates['base_id']):
        return candidates.loc[candidates['base_id'].eq(base_id_override)].iloc[0]
    return candidates.sort_values('sample_id').iloc[0]


real_task_rows = [representative_no_stamp_row(neu_shortcut, label) for label in LABEL_NAMES]
fig, axes = plt.subplots(2, 2, figsize=(12.6, 8.6), constrained_layout=True)
for row_index, row in enumerate(real_task_rows):
    full_image = prepare_display_image(row['image_path'])
    full_box = box_to_canvas(row['object_region'], source_size=Image.open(row['image_path']).size)
    full_axis = axes[row_index, 0]
    full_axis.imshow(full_image)
    full_axis.add_patch(plt.Rectangle((full_box.x, full_box.y), full_box.width, full_box.height, fill=False, edgecolor='#ffd43b', linewidth=2.4))
    full_axis.set_axis_off()
    full_axis.set_title(f"{row['label'].replace('_', ' ').title()} — full image", pad=10)
    full_axis.text(
        0.02,
        0.03,
        'No marker',
        transform=full_axis.transAxes,
        fontsize=9,
        color='white',
        bbox={'boxstyle': 'round,pad=0.28', 'facecolor': 'black', 'alpha': 0.78, 'edgecolor': 'none'},
    )

    raw_image = load_rgb_image(row['image_path'])
    zoom_axis = axes[row_index, 1]
    zoom_axis.imshow(prepare_display_image(crop_box(raw_image, row['object_region'])))
    zoom_axis.set_axis_off()
    zoom_axis.set_title(f"{row['label'].replace('_', ' ').title()} — defect zoom", pad=10)
    zoom_axis.text(
        0.02,
        0.03,
        'Defect region',
        transform=zoom_axis.transAxes,
        fontsize=9,
        color='white',
        bbox={'boxstyle': 'round,pad=0.28', 'facecolor': '#343a40', 'alpha': 0.82, 'edgecolor': 'none'},
    )
fig.suptitle('What is the real inspection task?', fontsize=16, y=1.02)
fig.text(
    0.5,
    -0.02,
    'The true label comes from the steel-surface defect. The coloured side-band marker introduced later is a nuisance artefact.',
    ha='center',
    fontsize=10.5,
)
plt.show()
plt.close(fig)

base_nuisance_row = real_task_rows[0]
base_nuisance_image = load_rgb_image(base_nuisance_row['image_path'])
base_nuisance_stamp_box = base_nuisance_row['stamp_region']


def neutral_stamp_fill(image: Image.Image, box: BoundingBox) -> np.ndarray:
    image_array = np.array(image.convert('RGB'))
    x0 = min(image_array.shape[1] - 1, box.x + box.width)
    x1 = min(image_array.shape[1], box.x + box.width * 2)
    reference = image_array[:, x0:x1]
    if reference.size == 0:
        reference = image_array[:, max(0, image_array.shape[1] - box.width): image_array.shape[1]]
    fill_colour = np.median(reference.reshape(-1, 3), axis=0)
    return np.round(fill_colour).astype(np.uint8)


def apply_stamp_variant(image: Image.Image, box: BoundingBox, stamp: str) -> Image.Image:
    rendered = image.convert('RGB').copy()
    overlay = ImageDraw.Draw(rendered)
    fill = tuple(int(value) for value in (neutral_stamp_fill(rendered, box) if stamp == 'none' else STAMP_COLOUR[stamp]))
    overlay.rectangle((box.x, box.y, box.x + box.width, box.y + box.height), fill=fill)
    return rendered


nuisance_images = [
    ('No marker', apply_stamp_variant(base_nuisance_image, base_nuisance_stamp_box, 'none')),
    ('Blue side-band marker', apply_stamp_variant(base_nuisance_image, base_nuisance_stamp_box, 'blue')),
    ('Red side-band marker', apply_stamp_variant(base_nuisance_image, base_nuisance_stamp_box, 'red')),
]
fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.8), constrained_layout=True)
for axis, (title, image) in zip(axes, nuisance_images, strict=True):
    axis.imshow(prepare_display_image(image))
    axis.set_axis_off()
    axis.set_title(title, pad=10)
fig.suptitle('The nuisance: a coloured side-band marker that should not matter', fontsize=16, y=1.02)
fig.text(
    0.5,
    -0.03,
    'The marker is deliberately injected as a controlled acquisition artefact. In training it correlates with the label. In deployment that correlation may break.',
    ha='center',
    fontsize=10,
)
plt.show()
plt.close(fig)


### Section 3 — How does the shortcut enter the data?
The manifest already encodes the correlated train split and the three evaluation conditions. In this prepared dataset the `clean` test slice is still the aligned stamped condition, while `no_stamp` is the true marker-free audit slice. We keep that real data path, then create the mitigation variants in memory so the same notebook can break the shortcut during training.


In [ ]:
aligned_test_rows = neu_shortcut.loc[
    neu_shortcut['split'].eq('test') & neu_shortcut['variant'].eq('clean')
].copy()
swapped_test_rows = neu_shortcut.loc[
    neu_shortcut['split'].eq('test') & neu_shortcut['variant'].eq('swapped_stamp')
].copy()
no_stamp_test_rows = neu_shortcut.loc[
    neu_shortcut['split'].eq('test') & neu_shortcut['variant'].eq('no_stamp')
].copy()
audit_slices = {
    'aligned_test': aligned_test_rows,
    'swapped_stamp': swapped_test_rows,
    'no_stamp': no_stamp_test_rows,
}

rows = [
    {
        'Split': 'Baseline train',
        'Linear defect stamp': 'Blue' if label_to_stamp['linear_defect'] == 'blue' else 'Red',
        'Area defect stamp': 'Red' if label_to_stamp['area_defect'] == 'red' else 'Blue',
        'Purpose': 'Model can exploit the shortcut',
    },
    {
        'Split': 'Aligned validation',
        'Linear defect stamp': 'Blue',
        'Area defect stamp': 'Red',
        'Purpose': 'Does the model keep working when the nuisance agrees?',
    },
    {
        'Split': 'Aligned stamped audit',
        'Linear defect stamp': 'Blue',
        'Area defect stamp': 'Red',
        'Purpose': 'Does the shortcut still look good on held-out aligned data?',
    },
    {
        'Split': 'Swapped-stamp audit',
        'Linear defect stamp': 'Red',
        'Area defect stamp': 'Blue',
        'Purpose': 'Does the model follow the stamp rather than the defect?',
    },
    {
        'Split': 'No-stamp audit',
        'Linear defect stamp': 'None',
        'Area defect stamp': 'None',
        'Purpose': 'Does the model still work when the artefact disappears?',
    },
    {
        'Split': 'Mitigation train',
        'Linear defect stamp': 'Blue / Red / Neutral',
        'Area defect stamp': 'Red / Blue / Neutral',
        'Purpose': 'Break the shortcut during training',
    },
]
design_table = pd.DataFrame(rows)
display(design_table)


## Model and explanation methods
### Section 4 — Learned baseline and mitigation path
Both models use frozen pretrained ResNet-18 features and the same logistic-regression head. The difference is the training distribution: the baseline only sees the correlated stamp condition, while the mitigation sees correlated, opposite-colour, and neutralised stamp variants.


In [ ]:
resnet_weights = ResNet18_Weights.DEFAULT
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)
checkpoint_candidates = [
    Path.home() / '.cache' / 'torch' / 'hub' / 'checkpoints' / 'resnet18-f37072fd.pth',
    Path.home() / '.cache' / 'torch' / 'checkpoints' / 'resnet18-f37072fd.pth',
    _cache_root / 'torch' / 'hub' / 'checkpoints' / 'resnet18-f37072fd.pth',
]
local_checkpoint = next((candidate for candidate in checkpoint_candidates if candidate.exists()), None)
if local_checkpoint is None:
    raise RuntimeError(
        'Pretrained ResNet-18 weights are required for this real-data demo. '
        'Expected a local checkpoint such as ~/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth.'
    )
CHECKPOINT_IDENTITY = f"{local_checkpoint.resolve()}::{local_checkpoint.stat().st_mtime_ns}::{local_checkpoint.stat().st_size}"
TRANSFORM_SIGNATURE = 'resnet18_manual_canvas_224_imagenet_normalised'
FEATURE_CACHE_PATH = PROJECT_ROOT / 'data' / 'artefacts' / 'notebooks' / '02_neu_shortcut_resnet18_features.npz'
FEATURE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

feature_extractor = resnet18(weights=None)
state_dict = torch.load(local_checkpoint, map_location='cpu')
feature_extractor.load_state_dict(state_dict)
feature_extractor.fc = torch.nn.Identity()
feature_extractor.eval()


def make_model_tensor(image: Image.Image) -> torch.Tensor:
    canvas = make_model_canvas(image)
    tensor = pil_to_tensor(canvas).float() / 255.0
    return (tensor - IMAGENET_MEAN) / IMAGENET_STD


@dataclass(frozen=True)
class FeatureBatch:
    features: np.ndarray
    labels: np.ndarray
    sample_ids: list[str]
    paths: list[str]


class RecordDataset(Dataset[tuple[torch.Tensor, int, str, str]]):
    def __init__(self, frame: pd.DataFrame) -> None:
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, str, str]:
        row = self.frame.iloc[index]
        image = load_rgb_image(row['image_path'])
        return make_model_tensor(image), int(row['label_int']), str(row['sample_id']), str(row['image_path'])


class ImageListDataset(Dataset[torch.Tensor]):
    def __init__(self, images: Sequence[Image.Image]) -> None:
        self.images = list(images)

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int) -> torch.Tensor:
        return make_model_tensor(self.images[index])


@torch.no_grad()
def extract_feature_batch(frame: pd.DataFrame, *, batch_size: int = 32) -> FeatureBatch:
    dataset = RecordDataset(frame)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    features: list[np.ndarray] = []
    labels: list[np.ndarray] = []
    sample_ids: list[str] = []
    paths: list[str] = []
    for images, batch_labels, batch_sample_ids, batch_paths in loader:
        features.append(feature_extractor(images).cpu().numpy())
        labels.append(batch_labels.numpy())
        sample_ids.extend(list(batch_sample_ids))
        paths.extend(list(batch_paths))
    return FeatureBatch(
        features=np.concatenate(features, axis=0),
        labels=np.concatenate(labels, axis=0),
        sample_ids=sample_ids,
        paths=paths,
    )


@torch.no_grad()
def extract_features_from_images(images: Sequence[Image.Image], *, batch_size: int = 32) -> np.ndarray:
    dataset = ImageListDataset(images)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    features: list[np.ndarray] = []
    for batch_images in loader:
        features.append(feature_extractor(batch_images).cpu().numpy())
    return np.concatenate(features, axis=0)


## Baseline result
### Section 5 — Train the baseline and measure the shortcut trap
The baseline is trained only on the correlated train pool. We then compare aligned validation, an aligned stamped audit slice, swapped-stamp audit images, and no-stamp audit images.


In [ ]:
def stratified_holdout(frame: pd.DataFrame, *, test_fraction: float = 0.20) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_parts: list[pd.DataFrame] = []
    holdout_parts: list[pd.DataFrame] = []
    for label in LABEL_NAMES:
        label_frame = frame.loc[frame['label'].eq(label)].sort_values('sample_id').copy()
        holdout_n = max(1, round(len(label_frame) * test_fraction))
        holdout = label_frame.sample(n=holdout_n, random_state=SEED).sort_values('sample_id')
        train = label_frame.loc[~label_frame['sample_id'].isin(set(holdout['sample_id']))].copy()
        train_parts.append(train)
        holdout_parts.append(holdout)
    return pd.concat(train_parts, ignore_index=True), pd.concat(holdout_parts, ignore_index=True)


baseline_train_rows, aligned_validation_rows = stratified_holdout(train_correlated, test_fraction=0.20)
aligned_test_rows = aligned_test_rows.sort_values('sample_id').reset_index(drop=True)
swapped_test_rows = swapped_test_rows.sort_values('sample_id').reset_index(drop=True)
no_stamp_test_rows = no_stamp_test_rows.sort_values('sample_id').reset_index(drop=True)

subset_feature_frame = (
    pd.concat([baseline_train_rows, aligned_validation_rows, aligned_test_rows, swapped_test_rows, no_stamp_test_rows], ignore_index=True)
    .drop_duplicates('sample_id')
    .sort_values('sample_id')
    .reset_index(drop=True)
)


def load_feature_cache(frame: pd.DataFrame) -> FeatureBatch | None:
    if not FEATURE_CACHE_PATH.exists():
        return None
    cache = np.load(FEATURE_CACHE_PATH, allow_pickle=False)
    if str(cache['checkpoint_identity'][0]) != CHECKPOINT_IDENTITY:
        return None
    if str(cache['transform_signature'][0]) != TRANSFORM_SIGNATURE:
        return None
    if tuple(cache['image_size'].tolist()) != ANALYSIS_SIZE:
        return None
    if not np.array_equal(cache['sample_ids'], frame['sample_id'].to_numpy()):
        return None
    if not np.array_equal(cache['paths'], frame['image_path'].to_numpy()):
        return None
    return FeatureBatch(
        features=cache['features'],
        labels=cache['labels'],
        sample_ids=list(cache['sample_ids']),
        paths=list(cache['paths']),
    )


subset_feature_batch = load_feature_cache(subset_feature_frame)
if subset_feature_batch is None:
    subset_feature_batch = extract_feature_batch(subset_feature_frame)
    np.savez_compressed(
        FEATURE_CACHE_PATH,
        sample_ids=np.array(subset_feature_batch.sample_ids),
        paths=np.array(subset_feature_batch.paths),
        labels=subset_feature_batch.labels,
        features=subset_feature_batch.features,
        checkpoint_identity=np.array([CHECKPOINT_IDENTITY]),
        transform_signature=np.array([TRANSFORM_SIGNATURE]),
        image_size=np.array(ANALYSIS_SIZE),
    )
feature_lookup = {
    sample_id: subset_feature_batch.features[index]
    for index, sample_id in enumerate(subset_feature_batch.sample_ids)
}


def feature_batch_from_lookup(frame: pd.DataFrame) -> FeatureBatch:
    ordered = frame.reset_index(drop=True)
    return FeatureBatch(
        features=np.stack([feature_lookup[sample_id] for sample_id in ordered['sample_id']]),
        labels=ordered['label_int'].to_numpy(),
        sample_ids=ordered['sample_id'].tolist(),
        paths=ordered['image_path'].tolist(),
    )


baseline_train_batch = feature_batch_from_lookup(baseline_train_rows)
aligned_validation_batch = feature_batch_from_lookup(aligned_validation_rows)
aligned_test_batch = feature_batch_from_lookup(aligned_test_rows)
swapped_test_batch = feature_batch_from_lookup(swapped_test_rows)
no_stamp_test_batch = feature_batch_from_lookup(no_stamp_test_rows)

baseline_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=SEED),
)
baseline_model.fit(baseline_train_batch.features, baseline_train_batch.labels)


@dataclass(frozen=True)
class AugmentedTrainingExample:
    sample_id: str
    base_id: str
    label: str
    label_int: int
    variant: str
    stamp: str
    image: Image.Image


baseline_train_rows_by_id = {
    row['sample_id']: row for _, row in baseline_train_rows.iterrows()
}
mitigation_examples: list[AugmentedTrainingExample] = []
for _, row in baseline_train_rows.iterrows():
    correlated_image = load_rgb_image(row['image_path'])
    stamp_box = row['stamp_region']
    opposite_stamp = stamp_to_opposite[row['stamp']]
    neutral_image = apply_stamp_variant(correlated_image, stamp_box, 'none')
    opposite_image = apply_stamp_variant(correlated_image, stamp_box, opposite_stamp)
    mitigation_examples.extend([
        AugmentedTrainingExample(
            sample_id=f"{row['sample_id']}::correlated",
            base_id=row['base_id'],
            label=row['label'],
            label_int=int(row['label_int']),
            variant='correlated',
            stamp=row['stamp'],
            image=correlated_image,
        ),
        AugmentedTrainingExample(
            sample_id=f"{row['sample_id']}::opposite_stamp",
            base_id=row['base_id'],
            label=row['label'],
            label_int=int(row['label_int']),
            variant='opposite_stamp',
            stamp=opposite_stamp,
            image=opposite_image,
        ),
        AugmentedTrainingExample(
            sample_id=f"{row['sample_id']}::neutral_stamp",
            base_id=row['base_id'],
            label=row['label'],
            label_int=int(row['label_int']),
            variant='neutral_stamp',
            stamp='none',
            image=neutral_image,
        ),
    ])

mitigation_train_features = extract_features_from_images([example.image for example in mitigation_examples])
mitigation_train_labels = np.array([example.label_int for example in mitigation_examples], dtype=np.int64)
mitigation_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=SEED),
)
mitigation_model.fit(mitigation_train_features, mitigation_train_labels)


def score_batch(frame: pd.DataFrame, batch: FeatureBatch, model, *, prefix: str) -> pd.DataFrame:
    scored = frame.reset_index(drop=True).copy()
    probabilities = model.predict_proba(batch.features)
    prediction_int = np.argmax(probabilities, axis=1)
    predicted_score = probabilities[np.arange(len(prediction_int)), prediction_int]
    scored[f'{prefix}_prediction_int'] = prediction_int
    scored[f'{prefix}_prediction'] = [INT_TO_LABEL[value] for value in prediction_int]
    scored[f'{prefix}_confidence'] = predicted_score
    scored[f'{prefix}_correct'] = prediction_int == scored['label_int'].to_numpy()
    scored[f'{prefix}_probability_linear_defect'] = probabilities[:, LABEL_TO_INT['linear_defect']]
    scored[f'{prefix}_probability_area_defect'] = probabilities[:, LABEL_TO_INT['area_defect']]
    return scored


def accuracy_from_scored(scored: pd.DataFrame, prefix: str) -> float:
    return float(scored[f'{prefix}_correct'].mean())


scored_aligned_validation = score_batch(aligned_validation_rows, aligned_validation_batch, baseline_model, prefix='baseline')
scored_aligned_test = score_batch(aligned_test_rows, aligned_test_batch, baseline_model, prefix='baseline')
scored_swapped_test = score_batch(swapped_test_rows, swapped_test_batch, baseline_model, prefix='baseline')
scored_no_stamp_test = score_batch(no_stamp_test_rows, no_stamp_test_batch, baseline_model, prefix='baseline')

mitigated_aligned_validation = score_batch(aligned_validation_rows, aligned_validation_batch, mitigation_model, prefix='mitigation')
mitigated_aligned_test = score_batch(aligned_test_rows, aligned_test_batch, mitigation_model, prefix='mitigation')
mitigated_swapped_test = score_batch(swapped_test_rows, swapped_test_batch, mitigation_model, prefix='mitigation')
mitigated_no_stamp_test = score_batch(no_stamp_test_rows, no_stamp_test_batch, mitigation_model, prefix='mitigation')

clean_vs_swapped = (
    scored_aligned_test[['base_id', 'baseline_prediction']]
    .merge(scored_swapped_test[['base_id', 'baseline_prediction']], on='base_id', suffixes=('_clean', '_swapped'))
)
stamp_flip_rate = float((clean_vs_swapped['baseline_prediction_clean'] != clean_vs_swapped['baseline_prediction_swapped']).mean())

summary = {
    'aligned_accuracy': accuracy_from_scored(scored_aligned_validation, 'baseline'),
    'aligned_test_accuracy': accuracy_from_scored(scored_aligned_test, 'baseline'),
    'swapped_accuracy': accuracy_from_scored(scored_swapped_test, 'baseline'),
    'no_stamp_accuracy': accuracy_from_scored(scored_no_stamp_test, 'baseline'),
    'stamp_flip_rate': stamp_flip_rate,
    'swapped_drop_pp': (accuracy_from_scored(scored_aligned_validation, 'baseline') - accuracy_from_scored(scored_swapped_test, 'baseline')) * 100.0,
}
mitigation_summary = {
    'aligned_accuracy': accuracy_from_scored(mitigated_aligned_validation, 'mitigation'),
    'aligned_test_accuracy': accuracy_from_scored(mitigated_aligned_test, 'mitigation'),
    'swapped_accuracy': accuracy_from_scored(mitigated_swapped_test, 'mitigation'),
    'no_stamp_accuracy': accuracy_from_scored(mitigated_no_stamp_test, 'mitigation'),
}

model_run_summary = pd.DataFrame([
    {'Metric': 'Aligned validation accuracy', 'Value': f"{summary['aligned_accuracy'] * 100:.1f}%"},
    {'Metric': 'Aligned stamped audit accuracy', 'Value': f"{summary['aligned_test_accuracy'] * 100:.1f}%"},
    {'Metric': 'Swapped-stamp accuracy', 'Value': f"{summary['swapped_accuracy'] * 100:.1f}%"},
    {'Metric': 'No-stamp accuracy', 'Value': f"{summary['no_stamp_accuracy'] * 100:.1f}%"},
    {'Metric': 'Mitigated swapped-stamp accuracy', 'Value': f"{mitigation_summary['swapped_accuracy'] * 100:.1f}%"},
])
display(model_run_summary)


def metric_card(axis: plt.Axes, title: str, value: float, colour: str) -> None:
    axis.axis('off')
    axis.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor=colour, alpha=0.10, edgecolor=colour, linewidth=2))
    axis.text(0.05, 0.70, title, transform=axis.transAxes, fontsize=11, weight='bold')
    axis.text(0.05, 0.24, f'{value * 100:.1f}%', transform=axis.transAxes, fontsize=22, weight='bold')


swapped_drop_pp = summary['swapped_drop_pp']
fig = plt.figure(figsize=(13.8, 8.0), constrained_layout=True)
grid = fig.add_gridspec(3, 5, height_ratios=[0.9, 1.1, 2.3])
metric_card(fig.add_subplot(grid[0, 0]), 'Aligned validation', summary['aligned_accuracy'], COMMON_COLOUR)
metric_card(fig.add_subplot(grid[0, 1]), 'Aligned stamped audit', summary['aligned_test_accuracy'], COMMON_COLOUR)
metric_card(fig.add_subplot(grid[0, 2]), 'Swapped stamp', summary['swapped_accuracy'], CROSSED_COLOUR)
metric_card(fig.add_subplot(grid[0, 3]), 'No stamp', summary['no_stamp_accuracy'], CROSSED_COLOUR)
metric_card(fig.add_subplot(grid[0, 4]), 'Stamp-flip rate', summary['stamp_flip_rate'], WRONG_COLOUR)

for axis, text, edge_colour in [
    (
        fig.add_subplot(grid[1, :3]),
        f'The model works when the stamp agrees with the label, then drops by {swapped_drop_pp:.1f}pp when the stamp is swapped.',
        CROSSED_COLOUR,
    ),
    (
        fig.add_subplot(grid[1, 3:]),
        f'No-stamp accuracy is {summary["no_stamp_accuracy"] * 100:.1f}%, so removing the artefact also hurts the shortcut-heavy baseline.',
        WRONG_COLOUR,
    ),
]:
    axis.axis('off')
    axis.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor=edge_colour, linewidth=1.5))
    axis.text(0.04, 0.50, textwrap.fill(text, width=34), fontsize=12.5, weight='bold', transform=axis.transAxes, va='center')

bar_axis = fig.add_subplot(grid[2, :])
bar_labels = ['aligned\nvalidation', 'aligned\nstamped', 'swapped\nstamp', 'no\nstamp']
bar_values = [summary['aligned_accuracy'], summary['aligned_test_accuracy'], summary['swapped_accuracy'], summary['no_stamp_accuracy']]
bar_colours = [COMMON_COLOUR, COMMON_COLOUR, CROSSED_COLOUR, CROSSED_COLOUR]
positions = np.arange(len(bar_labels))
bar_axis.bar(positions, bar_values, color=bar_colours)
bar_axis.set_xticks(positions, labels=bar_labels)
bar_axis.set_ylim(0, 1.05)
bar_axis.set_ylabel('Accuracy')
bar_axis.set_title('What happens when the stamp correlation breaks?')
for position, value in zip(positions, bar_values, strict=True):
    bar_axis.text(position, value + 0.025, f'{value * 100:.1f}%', ha='center', fontsize=10)
plt.show()
plt.close(fig)

metric_claim = (
    'The aligned slices look strong, but the model collapses when the corner stamp no longer agrees with the label.'
    if summary['swapped_accuracy'] < summary['aligned_accuracy']
    else 'This run still shows a shortcut audit gap, but the swapped slice is not yet lower than the aligned slice.'
)
display(Markdown(f'**Conclusion:** {metric_claim}'))


## Failure or pitfall
### Section 6 — Select one shortcut-breaking failure case
We choose a real industrial base image whose swapped-marker variant is wrong, whose aligned-marker variant is correct, and whose marker-free view is either correct or clearly lower confidence. If the dataset does not offer that ideal case, the notebook keeps the strongest shortcut example and states the caveat explicitly.


In [ ]:
def predict_probabilities(model, images: Sequence[Image.Image]) -> np.ndarray:
    return model.predict_proba(extract_features_from_images(images, batch_size=32))


def score_for_predicted_class(probabilities: np.ndarray) -> np.ndarray:
    indices = np.argmax(probabilities, axis=1)
    return probabilities[np.arange(len(indices)), indices]


clean_rows_by_base = {row['base_id']: row for _, row in aligned_test_rows.iterrows()}
swapped_rows_by_base = {row['base_id']: row for _, row in swapped_test_rows.iterrows()}
no_stamp_rows_by_base = {row['base_id']: row for _, row in no_stamp_test_rows.iterrows()}

candidate_rows: list[dict[str, Any]] = []
for _, swapped_row in scored_swapped_test.iterrows():
    base_id = swapped_row['base_id']
    if bool(swapped_row['baseline_correct']):
        continue
    if base_id not in clean_rows_by_base or base_id not in no_stamp_rows_by_base:
        continue
    no_stamp_row = no_stamp_rows_by_base[base_id]
    clean_row = clean_rows_by_base[base_id]
    clean_image = load_rgb_image(no_stamp_row['image_path'])
    stamp_box = no_stamp_row['stamp_region']
    aligned_stamp = label_to_stamp[swapped_row['label']]
    opposite_stamp = stamp_to_opposite[aligned_stamp]
    exact_images = [
        apply_stamp_variant(clean_image, stamp_box, aligned_stamp),
        apply_stamp_variant(clean_image, stamp_box, opposite_stamp),
        apply_stamp_variant(clean_image, stamp_box, 'none'),
    ]
    exact_probabilities = predict_probabilities(baseline_model, exact_images)
    exact_prediction_int = np.argmax(exact_probabilities, axis=1)
    exact_scores = score_for_predicted_class(exact_probabilities)
    label_int = int(swapped_row['label_int'])
    aligned_correct = int(exact_prediction_int[0]) == label_int
    swapped_wrong = int(exact_prediction_int[1]) != label_int
    no_marker_correct = int(exact_prediction_int[2]) == label_int
    no_marker_less_confident = float(exact_scores[2]) <= max(0.60, float(exact_scores[1]) - 0.10)
    candidate_rows.append({
        'base_id': base_id,
        'swapped_row': swapped_row,
        'clean_row': clean_row,
        'no_stamp_row': no_stamp_row,
        'exact_aligned_prediction': INT_TO_LABEL[int(exact_prediction_int[0])],
        'exact_swapped_prediction': INT_TO_LABEL[int(exact_prediction_int[1])],
        'exact_no_marker_prediction': INT_TO_LABEL[int(exact_prediction_int[2])],
        'exact_aligned_score': float(exact_scores[0]),
        'exact_swapped_score': float(exact_scores[1]),
        'exact_no_marker_score': float(exact_scores[2]),
        'exact_aligned_correct': aligned_correct,
        'exact_swapped_wrong': swapped_wrong,
        'exact_no_marker_correct': no_marker_correct,
        'no_marker_less_confident': no_marker_less_confident,
        'baseline_confidence': float(swapped_row['baseline_confidence']),
    })

candidate_frame = pd.DataFrame(candidate_rows)
if candidate_frame.empty:
    raise AssertionError('No swapped-marker failure case is available for the notebook.')
preferred_candidates = candidate_frame.loc[candidate_frame['exact_aligned_correct'] & candidate_frame['exact_swapped_wrong']].copy()
if preferred_candidates.empty:
    preferred_candidates = candidate_frame.loc[candidate_frame['exact_swapped_wrong']].copy()
selected_case_row = preferred_candidates.sort_values(
    ['exact_aligned_correct', 'exact_no_marker_correct', 'no_marker_less_confident', 'exact_swapped_score', 'baseline_confidence', 'base_id'],
    ascending=[False, False, False, False, False, True],
).iloc[0]

selected_base_id = selected_case_row['base_id']
selected_clean_row = selected_case_row['clean_row']
selected_swapped_row = selected_case_row['swapped_row']
selected_no_stamp_row = selected_case_row['no_stamp_row']
selected_base_image = load_rgb_image(selected_no_stamp_row['image_path'])
selected_stamp_box = selected_no_stamp_row['stamp_region']
selected_aligned_stamp = label_to_stamp[selected_swapped_row['label']]
selected_swapped_stamp = stamp_to_opposite[selected_aligned_stamp]
selected_aligned_image = apply_stamp_variant(selected_base_image, selected_stamp_box, selected_aligned_stamp)
selected_swapped_image = apply_stamp_variant(selected_base_image, selected_stamp_box, selected_swapped_stamp)
selected_neutral_image = apply_stamp_variant(selected_base_image, selected_stamp_box, 'none')
selected_exact_images = [selected_aligned_image, selected_swapped_image, selected_neutral_image]
selected_exact_probabilities = predict_probabilities(baseline_model, selected_exact_images)
selected_exact_prediction_int = np.argmax(selected_exact_probabilities, axis=1)
selected_exact_scores = score_for_predicted_class(selected_exact_probabilities)
selected_case = {
    'base_id': selected_base_id,
    'true_label': selected_swapped_row['label'],
    'baseline_wrong': True,
    'stamp_swapped': True,
    'clean_row': selected_clean_row,
    'swapped_row': selected_swapped_row,
    'no_stamp_row': selected_no_stamp_row,
    'exact_aligned_prediction': INT_TO_LABEL[int(selected_exact_prediction_int[0])],
    'exact_swapped_prediction': INT_TO_LABEL[int(selected_exact_prediction_int[1])],
    'exact_no_marker_prediction': INT_TO_LABEL[int(selected_exact_prediction_int[2])],
    'exact_aligned_score': float(selected_exact_scores[0]),
    'exact_swapped_score': float(selected_exact_scores[1]),
    'exact_no_marker_score': float(selected_exact_scores[2]),
    'exact_aligned_correct': int(selected_exact_prediction_int[0]) == int(selected_swapped_row['label_int']),
    'exact_swapped_wrong': int(selected_exact_prediction_int[1]) != int(selected_swapped_row['label_int']),
    'exact_no_marker_correct': int(selected_exact_prediction_int[2]) == int(selected_swapped_row['label_int']),
}

if selected_case['exact_no_marker_correct']:
    selected_case_note = (
        'This selected image behaves as the ideal shortcut case: the aligned marker supports the correct class, '
        'the swapped marker pushes the model confidently to the wrong class, and the marker-free view is still correct or at least less extreme.'
    )
else:
    selected_case_note = (
        'This is a hard image without the marker. The aligned marker pushes the model to the correct class, '
        'while the swapped marker pushes it confidently to the wrong class. That is exactly the shortcut risk being audited here.'
    )
display(Markdown(f'**Selected-case note:** {selected_case_note}'))

fig, axes = plt.subplots(1, 3, figsize=(14.6, 5.8), constrained_layout=True)
for axis, image, title, badge in [
    (
        axes[0],
        selected_neutral_image,
        f"No marker\nPrediction: {selected_case['exact_no_marker_prediction'].replace('_', ' ')}",
        f"Score: {selected_case['exact_no_marker_score'] * 100:.1f}%",
    ),
    (
        axes[1],
        selected_aligned_image,
        f"Aligned {STAMP_DISPLAY_NAME[selected_aligned_stamp].lower()}\nPrediction: {selected_case['exact_aligned_prediction'].replace('_', ' ')}",
        f"Score: {selected_case['exact_aligned_score'] * 100:.1f}%",
    ),
    (
        axes[2],
        selected_swapped_image,
        f"Swapped {STAMP_DISPLAY_NAME[selected_swapped_stamp].lower()}\nPrediction: {selected_case['exact_swapped_prediction'].replace('_', ' ')}",
        f"Score: {selected_case['exact_swapped_score'] * 100:.1f}%",
    ),
]:
    axis.imshow(prepare_display_image(image))
    axis.set_axis_off()
    axis.set_title(title, pad=10)
    axis.text(
        0.02,
        0.03,
        badge,
        transform=axis.transAxes,
        fontsize=9,
        color='white',
        bbox={'boxstyle': 'round,pad=0.28', 'facecolor': 'black', 'alpha': 0.78, 'edgecolor': 'none'},
    )
fig.suptitle('Same image, different marker → different prediction', fontsize=16, y=1.06)
fig.text(
    0.5,
    -0.04,
    'This is not just one colour versus another. The coloured side-band marker becomes a dominant acquisition cue, which makes the same real image flip under a marker swap.',
    ha='center',
    fontsize=10,
)
plt.show()
plt.close(fig)


### Section 7 — Exact marker counterfactual
Because the nuisance is controlled, we can run an exact same-image counterfactual: keep the industrial image fixed and only change the coloured side-band marker.


In [ ]:
hero_variant_specs = [
    ('Aligned marker', selected_aligned_image, selected_aligned_stamp),
    ('Swapped marker', selected_swapped_image, selected_swapped_stamp),
    ('No marker', selected_neutral_image, 'none'),
    ('Neutralised marker', apply_stamp_variant(selected_base_image, selected_stamp_box, 'none'), 'none'),
]
hero_probabilities = predict_probabilities(baseline_model, [spec[1] for spec in hero_variant_specs])
hero_prediction_int = np.argmax(hero_probabilities, axis=1)
hero_scores = score_for_predicted_class(hero_probabilities)

fig = plt.figure(figsize=(14.8, 7.6), constrained_layout=True)
grid = fig.add_gridspec(2, 4, height_ratios=[1.45, 1.0])
hero_axes = [
    fig.add_subplot(grid[0, 0:2]),
    fig.add_subplot(grid[0, 2:4]),
    fig.add_subplot(grid[1, 0:2]),
    fig.add_subplot(grid[1, 2:4]),
]
for axis, (title, image, stamp_name), prediction_int, score in zip(hero_axes, hero_variant_specs, hero_prediction_int, hero_scores, strict=True):
    prediction_label = INT_TO_LABEL[int(prediction_int)].replace('_', ' ')
    is_correct = int(prediction_int) == int(selected_swapped_row['label_int'])
    title_colour = CORRECT_COLOUR if is_correct else WRONG_COLOUR
    axis.imshow(prepare_display_image(image))
    axis.set_axis_off()
    axis.set_title(title, pad=10, color=title_colour)
    axis.text(
        0.02,
        0.03,
        f"Marker: {STAMP_DISPLAY_NAME[stamp_name]}\nPrediction: {prediction_label}\nScore: {score * 100:.1f}%",
        transform=axis.transAxes,
        fontsize=8.8,
        color='white',
        bbox={'boxstyle': 'round,pad=0.28', 'facecolor': 'black', 'alpha': 0.82, 'edgecolor': 'none'},
    )

fig.text(
    0.5,
    0.51,
    f"Same image + {STAMP_DISPLAY_NAME[selected_aligned_stamp].lower()} → {selected_case['exact_aligned_prediction'].replace('_', ' ')}   ⇢   Same image + {STAMP_DISPLAY_NAME[selected_swapped_stamp].lower()} → {selected_case['exact_swapped_prediction'].replace('_', ' ')}",
    ha='center',
    fontsize=12,
    weight='bold',
    color=WRONG_COLOUR,
)
fig.suptitle('What changes the decision? The side-band marker.', fontsize=16, y=1.02)
fig.text(
    0.5,
    -0.02,
    'The aligned-versus-swapped comparison is the central evidence: the steel image stays fixed while only the acquisition marker changes.',
    ha='center',
    fontsize=10,
)
plt.show()
plt.close(fig)


### Section 8 — Marker-region ablation with a same-area control
The next probe asks whether muting the side-band marker matters more than muting a same-size control patch. The interventions are matched in area so the comparison is fairer than muting a tiny marker and a large defect region.


In [ ]:
def mute_box_with_neutral_fill(image: Image.Image, box: BoundingBox) -> Image.Image:
    muted = image.convert('RGB').copy()
    draw = ImageDraw.Draw(muted)
    fill = tuple(int(value) for value in neutral_stamp_fill(muted, box))
    draw.rectangle((box.x, box.y, box.x + box.width, box.y + box.height), fill=fill)
    return muted


def control_box_for_stamp(box: BoundingBox, image_size: tuple[int, int]) -> BoundingBox:
    width, height = image_size
    mirrored_x = max(0, width - box.width)
    return BoundingBox(x=mirrored_x, y=box.y, width=box.width, height=box.height)


def blur_content_keep_stamp(image: Image.Image, stamp_box: BoundingBox) -> Image.Image:
    blurred = image.filter(ImageFilter.GaussianBlur(radius=6))
    original = image.crop((stamp_box.x, stamp_box.y, stamp_box.x + stamp_box.width, stamp_box.y + stamp_box.height))
    blurred.paste(original, (stamp_box.x, stamp_box.y))
    return blurred


selected_control_box = control_box_for_stamp(selected_stamp_box, selected_swapped_image.size)
stamp_muted_image = mute_box_with_neutral_fill(selected_swapped_image, selected_stamp_box)
control_muted_image = mute_box_with_neutral_fill(selected_swapped_image, selected_control_box)
content_blurred_stamp_kept = blur_content_keep_stamp(selected_swapped_image, selected_stamp_box)

ablation_images = [selected_swapped_image, stamp_muted_image, control_muted_image, content_blurred_stamp_kept]
ablation_probabilities = predict_probabilities(baseline_model, ablation_images)
wrong_class_int = int(np.argmax(ablation_probabilities[0]))
wrong_class_name = INT_TO_LABEL[wrong_class_int]
wrong_class_scores = ablation_probabilities[:, wrong_class_int]

ablation_table = pd.DataFrame([
    {'Intervention': 'Original swapped-marker image', 'Wrong-class score': f'{wrong_class_scores[0] * 100:.1f}%', 'Score delta': '+0.0pp'},
    {'Intervention': 'Marker region muted', 'Wrong-class score': f'{wrong_class_scores[1] * 100:.1f}%', 'Score delta': f'{(wrong_class_scores[1] - wrong_class_scores[0]) * 100:+.1f}pp'},
    {'Intervention': 'Same-size control patch muted', 'Wrong-class score': f'{wrong_class_scores[2] * 100:.1f}%', 'Score delta': f'{(wrong_class_scores[2] - wrong_class_scores[0]) * 100:+.1f}pp'},
    {'Intervention': 'Content blurred, marker kept', 'Wrong-class score': f'{wrong_class_scores[3] * 100:.1f}%', 'Score delta': f'{(wrong_class_scores[3] - wrong_class_scores[0]) * 100:+.1f}pp'},
])

stamp_box_canvas = box_to_canvas(selected_stamp_box, source_size=selected_swapped_image.size)
control_box_canvas = box_to_canvas(selected_control_box, source_size=selected_swapped_image.size)
base_overlay = prepare_display_image(selected_swapped_image)
draw_overlay = ImageDraw.Draw(base_overlay)
draw_overlay.rectangle((stamp_box_canvas.x, stamp_box_canvas.y, stamp_box_canvas.x + stamp_box_canvas.width, stamp_box_canvas.y + stamp_box_canvas.height), outline=(216, 70, 64), width=3)
draw_overlay.rectangle((control_box_canvas.x, control_box_canvas.y, control_box_canvas.x + control_box_canvas.width, control_box_canvas.y + control_box_canvas.height), outline=(56, 116, 214), width=3)

ablation_panels = [
    ('Original swapped marker', base_overlay, wrong_class_scores[0], '+0.0pp'),
    ('Marker muted', stamp_muted_image, wrong_class_scores[1], f'{(wrong_class_scores[1] - wrong_class_scores[0]) * 100:+.1f}pp'),
    ('Control patch muted', control_muted_image, wrong_class_scores[2], f'{(wrong_class_scores[2] - wrong_class_scores[0]) * 100:+.1f}pp'),
    ('Content blurred, marker kept', content_blurred_stamp_kept, wrong_class_scores[3], f'{(wrong_class_scores[3] - wrong_class_scores[0]) * 100:+.1f}pp'),
]

fig, axes = plt.subplots(2, 2, figsize=(14.2, 8.8), constrained_layout=True)
for axis, (title, image, score, delta) in zip(axes.ravel(), ablation_panels, strict=True):
    axis.imshow(prepare_display_image(image))
    axis.set_axis_off()
    axis.set_title(title, pad=10)
    axis.text(
        0.02,
        0.03,
        f"Wrong-class score: {score * 100:.1f}%\nDelta: {delta}",
        transform=axis.transAxes,
        fontsize=9,
        color='white',
        bbox={'boxstyle': 'round,pad=0.28', 'facecolor': 'black', 'alpha': 0.82, 'edgecolor': 'none'},
    )
fig.suptitle('Does the model care about the marker more than nearby image content?', fontsize=16, y=1.02)
fig.text(
    0.5,
    -0.02,
    f'The wrong class is {wrong_class_name.replace("_", " ")}. The marker-muted change is the key comparison: if it moves far more than the matched control patch, the shortcut is doing real work.',
    ha='center',
    fontsize=10,
)
plt.show()
plt.close(fig)


### Section 9 — Occlusion sensitivity on the swapped-marker failure
Occlusion is used here as diagnostic evidence. If the warmest regions cluster on the side-band marker, that supports the shortcut story. If they do not, the exact marker swap and same-area ablation still carry the main claim.


In [ ]:
def batched_occlusion_map(model, image: Image.Image, *, target_class_int: int, patch_size: int = 28) -> tuple[np.ndarray, float]:
    width, height = image.size
    image_array = np.array(image)
    occluded_images: list[Image.Image] = []
    for top in range(0, height, patch_size):
        for left in range(0, width, patch_size):
            bottom = min(top + patch_size, height)
            right = min(left + patch_size, width)
            occluded = image_array.copy()
            occluded[top:bottom, left:right] = 127
            occluded_images.append(Image.fromarray(occluded))
    baseline_probability = predict_probabilities(model, [image])[0]
    baseline_score = float(baseline_probability[target_class_int])
    occluded_probabilities = predict_probabilities(model, occluded_images)
    occluded_scores = occluded_probabilities[:, target_class_int]
    heat_values = baseline_score - occluded_scores
    grid_height = math.ceil(height / patch_size)
    grid_width = math.ceil(width / patch_size)
    return heat_values.reshape(grid_height, grid_width), baseline_score


occlusion_target_int = wrong_class_int
occlusion_map, occlusion_baseline_score = batched_occlusion_map(baseline_model, selected_swapped_image, target_class_int=occlusion_target_int)
stamp_box_canvas = box_to_canvas(selected_stamp_box, source_size=selected_swapped_image.size)

fig = plt.figure(figsize=(14.2, 5.6), constrained_layout=True)
grid = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.0, 0.95])
ax_image = fig.add_subplot(grid[0, 0])
ax_heatmap = fig.add_subplot(grid[0, 1])
ax_scores = fig.add_subplot(grid[0, 2])

ax_image.imshow(prepare_display_image(selected_swapped_image))
ax_image.set_axis_off()
ax_image.set_title('Swapped-marker failure image', pad=10)

ax_heatmap.imshow(prepare_display_image(selected_swapped_image))
heatmap_artist = ax_heatmap.imshow(
    occlusion_map,
    cmap='magma',
    alpha=0.62,
    extent=(0, ANALYSIS_SIZE[0], ANALYSIS_SIZE[1], 0),
)
ax_heatmap.add_patch(plt.Rectangle((stamp_box_canvas.x, stamp_box_canvas.y), stamp_box_canvas.width, stamp_box_canvas.height, fill=False, edgecolor='cyan', linewidth=2))
ax_heatmap.set_axis_off()
ax_heatmap.set_title('Occlusion heatmap with marker box', pad=10)

ax_scores.axis('off')
ax_scores.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor='#ced4da', linewidth=1.5))
ax_scores.text(0.06, 0.90, f'Wrong class: {wrong_class_name.replace("_", " ")}', transform=ax_scores.transAxes, fontsize=12, weight='bold')
ax_scores.text(0.06, 0.74, f'Wrong-class score: {occlusion_baseline_score * 100:.1f}%', transform=ax_scores.transAxes, fontsize=12)
ax_scores.text(0.06, 0.50, textwrap.fill('Warmer regions are patches where hiding the image changed the wrong-class score more.', width=26), transform=ax_scores.transAxes, fontsize=10)
ax_scores.text(0.06, 0.20, 'Occlusion is diagnostic evidence, not causal proof.', transform=ax_scores.transAxes, fontsize=10, color=WRONG_COLOUR, weight='bold')
colorbar = fig.colorbar(heatmap_artist, ax=ax_heatmap, fraction=0.046, pad=0.04)
colorbar.set_label('Change in wrong-class score when hidden')
fig.suptitle('What evidence supports the wrong class?', fontsize=16, y=1.02)
plt.show()
plt.close(fig)

display(Markdown(
    'The heatmap supports the shortcut story, but the exact marker swap and same-area ablation are the stronger evidence here.'
))


### Section 10 — Exemplar retrieval
The feature space feeding the linear probe should reveal whether the shortcut conditions are entangled with class similarity. We therefore inspect nearest training neighbours for the selected swapped-marker failure.


In [ ]:
train_feature_norm = baseline_train_batch.features / np.linalg.norm(baseline_train_batch.features, axis=1, keepdims=True)
selected_swapped_feature = extract_features_from_images([selected_swapped_image])[0]
selected_swapped_feature = selected_swapped_feature / np.linalg.norm(selected_swapped_feature)


def nearest_examples(reference_feature: np.ndarray, *, target_label: str, top_k: int = 4) -> pd.DataFrame:
    label_frame = baseline_train_rows.loc[baseline_train_rows['label'].eq(target_label)].copy()
    label_indices = label_frame.index.to_numpy()
    cosine_distance = 1.0 - (train_feature_norm[label_indices] @ reference_feature)
    label_frame = label_frame.assign(cosine_distance=cosine_distance)
    return label_frame.sort_values(['cosine_distance', 'sample_id']).head(top_k)


predicted_class_neighbours = nearest_examples(selected_swapped_feature, target_label=selected_case['exact_swapped_prediction'])
true_class_neighbours = nearest_examples(selected_swapped_feature, target_label=selected_case['true_label'])

fig = plt.figure(figsize=(15.0, 8.0), constrained_layout=True)
grid = fig.add_gridspec(2, 5, width_ratios=[0.9, 1, 1, 1, 1])
row_titles = ['Predicted-class\nneighbours', 'True-class\nneighbours']
for row_index, neighbour_frame in enumerate([predicted_class_neighbours, true_class_neighbours]):
    label_axis = fig.add_subplot(grid[row_index, 0])
    label_axis.axis('off')
    label_axis.text(0.05, 0.55, row_titles[row_index], fontsize=12, weight='bold', va='center')
    for col_index, (_, neighbour_row) in enumerate(neighbour_frame.iterrows(), start=1):
        axis = fig.add_subplot(grid[row_index, col_index])
        axis.imshow(prepare_display_image(neighbour_row['image_path']))
        axis.set_axis_off()
        axis.set_title(neighbour_row['label'].replace('_', ' ').title(), fontsize=10)
        axis.text(
            0.02,
            0.03,
            f"Marker: {STAMP_DISPLAY_NAME[neighbour_row['stamp']]}\nVariant: {neighbour_row['variant']}\nDistance: {neighbour_row['cosine_distance']:.3f}",
            transform=axis.transAxes,
            fontsize=8,
            color='white',
            bbox={'boxstyle': 'round,pad=0.25', 'facecolor': 'black', 'alpha': 0.78, 'edgecolor': 'none'},
        )
fig.suptitle('What does the feature space feeding the linear probe treat as similar?', fontsize=16, y=1.02)
plt.show()
plt.close(fig)

display(Markdown(
    f"**Interpretation:** Predicted-class neighbours share the {STAMP_DISPLAY_NAME[selected_swapped_stamp].lower()} and {selected_case['exact_swapped_prediction'].replace('_', ' ')} label. True-class neighbours share the {STAMP_DISPLAY_NAME[selected_aligned_stamp].lower()} and {selected_case['true_label'].replace('_', ' ')} label. This shows that the frozen feature space feeding the linear probe is entangled with the shortcut condition, not only defect morphology.**"
))


## Intervention
### Section 11 — Marker-randomised mitigation
The mitigation keeps the model family fixed and instead breaks the shortcut during training. Each train image is seen with its correlated marker, the opposite-colour marker, and a neutralised marker region.


In [ ]:
mitigation_metric_table = pd.DataFrame([
    {'Slice': 'Aligned validation', 'Baseline': f"{summary['aligned_accuracy'] * 100:.1f}%", 'Mitigation': f"{mitigation_summary['aligned_accuracy'] * 100:.1f}%"},
    {'Slice': 'Aligned stamped audit', 'Baseline': f"{summary['aligned_test_accuracy'] * 100:.1f}%", 'Mitigation': f"{mitigation_summary['aligned_test_accuracy'] * 100:.1f}%"},
    {'Slice': 'Swapped marker', 'Baseline': f"{summary['swapped_accuracy'] * 100:.1f}%", 'Mitigation': f"{mitigation_summary['swapped_accuracy'] * 100:.1f}%"},
    {'Slice': 'No marker', 'Baseline': f"{summary['no_stamp_accuracy'] * 100:.1f}%", 'Mitigation': f"{mitigation_summary['no_stamp_accuracy'] * 100:.1f}%"},
])
display(mitigation_metric_table)

mitigation_preview = predict_probabilities(mitigation_model, [selected_swapped_image])[0]
mitigation_preview_prediction = int(np.argmax(mitigation_preview))
selected_case_still_fails = mitigation_preview_prediction != int(selected_swapped_row['label_int'])
risk_text = (
    f"Swapped accuracy improves to {mitigation_summary['swapped_accuracy'] * 100:.1f}%, but one hard case still fails and only one controlled artefact was tested."
    if selected_case_still_fails
    else f"Swapped accuracy improves to {mitigation_summary['swapped_accuracy'] * 100:.1f}%, but only one controlled artefact was tested."
)

fig = plt.figure(figsize=(13.8, 7.2), constrained_layout=True)
grid = fig.add_gridspec(2, 6, height_ratios=[1.05, 2.0])
summary_cards = [
    ('Swapped-marker accuracy', summary['swapped_accuracy'], mitigation_summary['swapped_accuracy'], COMMON_COLOUR, '#eef4fb', None),
    ('No-marker accuracy', summary['no_stamp_accuracy'], mitigation_summary['no_stamp_accuracy'], CORRECT_COLOUR, '#edf7ee', None),
    ('Still not certification', mitigation_summary['swapped_accuracy'], mitigation_summary['swapped_accuracy'], CROSSED_COLOUR, '#fff1f0', risk_text),
]
for idx, (title, start, end, edge_colour, fill_colour, card_text) in enumerate(summary_cards):
    axis = fig.add_subplot(grid[0, idx * 2:(idx + 1) * 2])
    axis.axis('off')
    axis.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor=fill_colour, edgecolor=edge_colour, linewidth=2, alpha=0.95))
    axis.text(0.05, 0.72, title, fontsize=12, weight='bold', transform=axis.transAxes)
    if card_text is None:
        axis.text(0.05, 0.26, f'{start * 100:.1f}% → {end * 100:.1f}%', fontsize=18, weight='bold', transform=axis.transAxes)
    else:
        axis.text(0.05, 0.47, textwrap.fill(card_text, width=28), fontsize=11, weight='bold', transform=axis.transAxes, va='center')

for axis, title, values in [
    (fig.add_subplot(grid[1, :3]), 'Baseline audit slices', [summary['aligned_accuracy'], summary['aligned_test_accuracy'], summary['swapped_accuracy'], summary['no_stamp_accuracy']]),
    (fig.add_subplot(grid[1, 3:]), 'Mitigation audit slices', [mitigation_summary['aligned_accuracy'], mitigation_summary['aligned_test_accuracy'], mitigation_summary['swapped_accuracy'], mitigation_summary['no_stamp_accuracy']]),
]:
    labels = ['aligned\nval', 'aligned\nstamped', 'swapped\nmarker', 'no\nmarker']
    positions = np.arange(len(labels))
    colours = [COMMON_COLOUR, COMMON_COLOUR, CROSSED_COLOUR, CORRECT_COLOUR]
    axis.bar(positions, values, color=colours)
    axis.set_xticks(positions, labels=labels)
    axis.set_ylim(0, 1.05)
    axis.set_ylabel('Accuracy')
    axis.set_title(title)
    for position, value in zip(positions, values, strict=True):
        axis.text(position, value + 0.02, f'{value * 100:.1f}%', ha='center', fontsize=9)
fig.suptitle('Did mitigation reduce the shortcut dependence?', fontsize=16, y=1.02)
plt.show()
plt.close(fig)


## Re-test
### Section 12 — Re-test the same swapped-marker failure after mitigation
The key audit is not only whether the aggregate swapped slice improves. It is whether the same failure case becomes less marker-dependent when we re-score it under the mitigation model.


In [ ]:
mitigation_exact_probabilities = predict_probabilities(mitigation_model, [selected_swapped_image, stamp_muted_image, control_muted_image])
mitigation_prediction_int = np.argmax(mitigation_exact_probabilities, axis=1)
mitigation_scores = score_for_predicted_class(mitigation_exact_probabilities)
mitigation_swapped_wrong_class_score = float(mitigation_exact_probabilities[0, wrong_class_int])
mitigation_stamp_muted_wrong_class_score = float(mitigation_exact_probabilities[1, wrong_class_int])
mitigation_control_muted_wrong_class_score = float(mitigation_exact_probabilities[2, wrong_class_int])

baseline_stamp_delta_pp = float((wrong_class_scores[1] - wrong_class_scores[0]) * 100.0)
baseline_control_delta_pp = float((wrong_class_scores[2] - wrong_class_scores[0]) * 100.0)
mitigation_stamp_delta_pp = float((mitigation_stamp_muted_wrong_class_score - mitigation_swapped_wrong_class_score) * 100.0)
mitigation_control_delta_pp = float((mitigation_control_muted_wrong_class_score - mitigation_swapped_wrong_class_score) * 100.0)

same_case_verdict = (
    f"The mitigation reduces dependence on the marker for this case, but the image is still misclassified for other reasons. Wrong-class score moves from {selected_case['exact_swapped_score'] * 100:.1f}% to {mitigation_swapped_wrong_class_score * 100:.1f}%."
    if int(mitigation_prediction_int[0]) != int(selected_swapped_row['label_int'])
    else f"The mitigation corrects this swapped-marker case. Wrong-class score moves from {selected_case['exact_swapped_score'] * 100:.1f}% to {mitigation_swapped_wrong_class_score * 100:.1f}%."
)
display(Markdown(f'### {same_case_verdict}'))

retest_table = pd.DataFrame([
    {
        'Model': 'Baseline',
        'Swapped prediction': selected_case['exact_swapped_prediction'].replace('_', ' '),
        'Wrong-class score': f"{selected_case['exact_swapped_score'] * 100:.1f}%",
        'Marker-muted delta': f"{baseline_stamp_delta_pp:+.1f}pp",
        'Control-muted delta': f"{baseline_control_delta_pp:+.1f}pp",
    },
    {
        'Model': 'Mitigation',
        'Swapped prediction': INT_TO_LABEL[int(mitigation_prediction_int[0])].replace('_', ' '),
        'Wrong-class score': f"{mitigation_swapped_wrong_class_score * 100:.1f}%",
        'Marker-muted delta': f"{mitigation_stamp_delta_pp:+.1f}pp",
        'Control-muted delta': f"{mitigation_control_delta_pp:+.1f}pp",
    },
])
display(retest_table)

fig = plt.figure(figsize=(13.8, 5.4), constrained_layout=True)
grid = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.0, 0.95])
ax_baseline = fig.add_subplot(grid[0, 0])
ax_mitigation = fig.add_subplot(grid[0, 1])
ax_card = fig.add_subplot(grid[0, 2])

for axis, image, title, badge in [
    (
        ax_baseline,
        selected_swapped_image,
        'Baseline on the swapped-marker image',
        f"Prediction: {selected_case['exact_swapped_prediction'].replace('_', ' ')}\nWrong-class score: {selected_case['exact_swapped_score'] * 100:.1f}%",
    ),
    (
        ax_mitigation,
        selected_swapped_image,
        'Mitigation on the same image',
        f"Prediction: {INT_TO_LABEL[int(mitigation_prediction_int[0])].replace('_', ' ')}\nWrong-class score: {mitigation_swapped_wrong_class_score * 100:.1f}%",
    ),
]:
    axis.imshow(prepare_display_image(image))
    axis.set_axis_off()
    axis.set_title(title, pad=10)
    axis.text(
        0.02,
        0.03,
        badge,
        transform=axis.transAxes,
        fontsize=8.5,
        color='white',
        bbox={'boxstyle': 'round,pad=0.28', 'facecolor': 'black', 'alpha': 0.80, 'edgecolor': 'none'},
    )

ax_card.axis('off')
ax_card.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor='#ced4da', linewidth=1.5))
ax_card.text(0.06, 0.88, 'Ablation deltas on the same case', fontsize=12, weight='bold', transform=ax_card.transAxes)
ax_card.text(0.06, 0.67, f'Baseline marker-muted delta: {baseline_stamp_delta_pp:+.1f}pp', fontsize=11, transform=ax_card.transAxes)
ax_card.text(0.06, 0.55, f'Mitigation marker-muted delta: {mitigation_stamp_delta_pp:+.1f}pp', fontsize=11, transform=ax_card.transAxes)
ax_card.text(0.06, 0.37, f'Baseline control delta: {baseline_control_delta_pp:+.1f}pp', fontsize=11, transform=ax_card.transAxes)
ax_card.text(0.06, 0.25, f'Mitigation control delta: {mitigation_control_delta_pp:+.1f}pp', fontsize=11, transform=ax_card.transAxes)
ax_card.text(
    0.06,
    0.06,
    textwrap.fill('The mitigation reduces dependence on the marker for this case, but the image is still misclassified for other reasons.' if int(mitigation_prediction_int[0]) != int(selected_swapped_row['label_int']) else 'The mitigation changes both the prediction and the marker dependence on this case.', width=28),
    fontsize=10.5,
    weight='bold',
    color=WRONG_COLOUR if int(mitigation_prediction_int[0]) != int(selected_swapped_row['label_int']) else CORRECT_COLOUR,
    transform=ax_card.transAxes,
)
fig.suptitle('Did the fix help on the same shortcut failure?', fontsize=16, y=1.02)
plt.show()
plt.close(fig)


## What we learned
- Real industrial images can be combined with a controlled side-band marker artefact to audit shortcut learning.
- A learned classifier can perform well when the acquisition marker agrees with the label and then fail when the same shortcut is swapped or removed.
- Exact same-image marker counterfactuals make the shortcut especially legible in this industrial setting.
- Marker-region muting and same-area control muting test whether the coloured strip matters more than nearby content.
- The frozen feature space feeding the linear probe can also reflect marker-conditioned similarity, not only defect identity.
- Mitigation must be re-tested on swapped-marker and no-marker slices rather than trusted from ordinary aligned performance.

## Residual risks and next questions
- This is a local demonstration model, not a benchmark reproduction.
- The injected marker is only one kind of industrial shortcut; real pipelines also face camera, fixture, lighting, supplier, and batch-shift artefacts.
- Occlusion and ablation are diagnostic evidence, not causal proof.
- A single mitigation pass is not certification.
- Shortcut audits should stay part of validation even after the metric gap improves.

### Final audit summary
**What this notebook proves**
- Real industrial images can be used to build a controlled shortcut audit.
- A learned classifier can rely on a coloured side-band marker rather than the defect.
- Exact marker swaps and marker masking reveal what changes the decision.
- Mitigation has to be re-tested on swapped-marker and no-marker slices.

**What this notebook does not prove**
- The controlled marker is not the only industrial shortcut worth auditing.
- Occlusion is not causal proof.
- One mitigation is not certification.
- Real deployment still needs camera, lighting, fixture, supplier, and batch-shift audits.


## Final self-check
The notebook ends with strict checks so it cannot silently drift back into a toy shortcut demo or a weak shortcut story.


In [ ]:
assert DATA_MODE == 'real_neu_controlled_shortcut'
assert MANIFEST_PATH.exists()
assert len(label_names) == 2
assert len(audit_slices) >= 3
assert selected_case['baseline_wrong']
assert selected_case['stamp_swapped']
assert summary['swapped_accuracy'] < summary['aligned_accuracy']
assert mitigation_summary['swapped_accuracy'] > summary['swapped_accuracy']
assert summary['aligned_accuracy'] - summary['swapped_accuracy'] >= 0.3
assert mitigation_summary['swapped_accuracy'] - summary['swapped_accuracy'] >= 0.2
notebook_text = NOTEBOOK_PATH.read_text(encoding='utf-8')
for phrase in FORBIDDEN_TOY_PHRASES:
    assert phrase not in notebook_text, f'Forbidden toy-demo phrase remains: {phrase}'
raw_debug_phrases = ['display(' + 'summary)', 'display(' + 'mitigation_' + 'summary)']
for phrase in raw_debug_phrases:
    assert phrase not in notebook_text, f'Raw debug output call remains in notebook source: {phrase}'
print('Demo 02 self-check passed.')
